# Chapter 8 — Convolutional Neural Networks (basics)

Runnable companion to `docs/07_cnn_basics.md`. Four exercises:

1. **Output-shape calculator** for `nn.Conv2d` — verify the formula by hand
   and against PyTorch.
2. **What does a convolution do?** Apply a hand-built 3×3 horizontal-edge
   kernel to a real MNIST digit and visualize the result.
3. **Build a SmallCNN** following the canonical `Conv → BN → ReLU → Pool`
   pattern and inspect its per-layer shapes.
4. **MLP vs. CNN on MNIST** at similar parameter count. The CNN should win.

Generated by `src/build_chapter_07_cnn_basics.py`. Edit the builder, not
this notebook.

## Setup

In [1]:
import math
from pathlib import Path as _Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cpu")

_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
MNIST_ROOT = str(_ROOT / "datasets" / "mnist")
print("torch", torch.__version__, "device", DEVICE)
print("mnist root:", MNIST_ROOT)

torch 2.12.0+cu130 device cpu
mnist root: /home/bangbc/Documents/AI_Courses/Deep_Learning_Foundation/datasets/mnist


## 1. Output-shape calculator for `nn.Conv2d`

Given input height `H`, kernel size `k`, padding `p`, stride `s`,
dilation `d`:

```
H_out = floor( (H + 2p − d · (k − 1) − 1) / s + 1 )
```

We verify the formula against PyTorch on three common patterns.

In [2]:
def conv_output_size(H, k, p, s, d=1):
    return (H + 2 * p - d * (k - 1) - 1) // s + 1


cases = [
    {"H": 28, "k": 3, "p": 1, "s": 1, "label": "same (k=3, p=1, s=1)"},
    {"H": 28, "k": 3, "p": 1, "s": 2, "label": "downsample x2 (k=3, p=1, s=2)"},
    {"H": 28, "k": 5, "p": 0, "s": 1, "label": "valid (k=5, p=0, s=1)"},
    {"H": 28, "k": 1, "p": 0, "s": 1, "label": "1x1 pointwise"},
]

print(f"{'pattern':<30s} {'formula':<10s} {'torch':<10s} match")
for c in cases:
    formula = conv_output_size(c["H"], c["k"], c["p"], c["s"])
    conv = nn.Conv2d(1, 1, kernel_size=c["k"], padding=c["p"], stride=c["s"])
    torch_out = conv(torch.zeros(1, 1, c["H"], c["H"])).shape[-1]
    print(f"{c['label']:<30s} {formula:<10d} {torch_out:<10d} {formula == torch_out}")

pattern                        formula    torch      match
same (k=3, p=1, s=1)           28         28         True
downsample x2 (k=3, p=1, s=2)  14         14         True
valid (k=5, p=0, s=1)          24         24         True
1x1 pointwise                  28         28         True


## 2. What does a convolution actually do?

A hand-built `3×3` horizontal-edge kernel detects up-down brightness
changes. We apply it (without learning) to a real MNIST digit.

In [3]:
mnist_train = datasets.MNIST(MNIST_ROOT, train=True, download=True, transform=None)
img_pil, label = mnist_train[7]
img = torch.tensor(np.array(img_pil), dtype=torch.float32) / 255.0
print(f"image shape: {tuple(img.shape)}  label: {label}")

# Sobel-like horizontal-edge detector.
kernel = torch.tensor([[-1., -2., -1.],
                       [ 0.,  0.,  0.],
                       [ 1.,  2.,  1.]])
edge = F.conv2d(img.view(1, 1, 28, 28),
                kernel.view(1, 1, 3, 3),
                padding=1).squeeze().abs()

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img,     cmap="gray");        axes[0].set_title("input image"); axes[0].axis("off")
axes[1].imshow(kernel,  cmap="bwr", vmin=-2, vmax=2)
axes[1].set_title("3x3 horizontal-edge kernel"); axes[1].axis("off")
axes[2].imshow(edge,    cmap="hot");          axes[2].set_title("|Conv2d output|"); axes[2].axis("off")
plt.tight_layout(); plt.show()

image shape: (28, 28)  label: 3


/tmp/ipykernel_209299/3980902310.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The same operation, applied at every spatial position with shared weights
— that is the entire idea of "parameter sharing + local connectivity". A
*trained* CNN learns dozens of such kernels jointly so the higher layers
can combine them into edges → corners → parts → objects.

## 3. Build a `SmallCNN`

The canonical stacking pattern from `docs/07_cnn_basics.md`:

```
Conv → BN → ReLU → MaxPool → ... → AdaptiveAvgPool → Flatten → Linear
```

We instantiate it for MNIST (1 input channel, 10 classes) and print the
output shape at every block boundary.

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
    def forward(self, x):
        return self.block(x)


class SmallCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc   = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


model = SmallCNN(in_channels=1, num_classes=10)
x = torch.randn(2, 1, 28, 28)

# Manually trace the shapes through each block.
print(f"input        : {tuple(x.shape)}")
y = model.features[0](x); print(f"after block 1: {tuple(y.shape)}")
y = model.features[1](y); print(f"after block 2: {tuple(y.shape)}")
y = model.features[2](y); print(f"after block 3: {tuple(y.shape)}")
y = model.pool(y);        print(f"after pool   : {tuple(y.shape)}")
y = y.flatten(1);         print(f"after flatten: {tuple(y.shape)}")
y = model.fc(y);          print(f"logits       : {tuple(y.shape)}")
print()
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

input        : (2, 1, 28, 28)
after block 1: (2, 16, 14, 14)
after block 2: (2, 32, 7, 7)
after block 3: (2, 64, 3, 3)
after pool   : (2, 64, 1, 1)
after flatten: (2, 64)
logits       : (2, 10)

params: 24,170


## 4. MLP vs. CNN on MNIST

Both models target the same parameter budget (~25–30k). We train each for
2 epochs on a 6,000-image MNIST subset and compare validation accuracy.

The CNN should win by a clear margin — that is the *inductive bias* of
local connectivity + parameter sharing paying off.

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_train = datasets.MNIST(MNIST_ROOT, train=True,  download=True, transform=transform)
mnist_val   = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(mnist_train, list(range(6000))),
                          batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(Subset(mnist_val,   list(range(2000))),
                          batch_size=256, shuffle=False, num_workers=0)


class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # ~25k params: 784 -> 32 -> 10
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 10),
        )
    def forward(self, x):
        return self.net(x)


def quick_train(model, num_epochs=2, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    train_losses, val_accs = [], []
    for _ in range(num_epochs):
        model.train()
        running, n = 0.0, 0
        for x, y in train_loader:
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            running += loss.item() * x.size(0); n += x.size(0)
        train_losses.append(running / n)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                pred = model(x).argmax(dim=1)
                correct += (pred == y).sum().item(); total += y.size(0)
        val_accs.append(correct / total)
    return train_losses, val_accs


torch.manual_seed(0)
mlp = TinyMLP()
mlp_train, mlp_val = quick_train(mlp)

torch.manual_seed(0)
cnn = SmallCNN(in_channels=1, num_classes=10)
cnn_train, cnn_val = quick_train(cnn)

print(f"MLP params: {sum(p.numel() for p in mlp.parameters()):,}")
print(f"CNN params: {sum(p.numel() for p in cnn.parameters()):,}")
print()
print(f"{'model':<6s}  {'epoch 1':<10s} {'epoch 2':<10s}")
print(f"{'MLP':<6s}  {mlp_val[0]:<10.4f} {mlp_val[1]:<10.4f}")
print(f"{'CNN':<6s}  {cnn_val[0]:<10.4f} {cnn_val[1]:<10.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([1, 2], mlp_val, marker="o", label="MLP (~25k params)")
ax.plot([1, 2], cnn_val, marker="o", label="SmallCNN (~24k params)")
ax.set_xticks([1, 2]); ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy")
ax.set_title("Same parameter budget, different architecture"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


MLP params: 25,450
CNN params: 24,170

model   epoch 1    epoch 2   
MLP     0.8375     0.8560    
CNN     0.8000     0.9225    


/tmp/ipykernel_209299/1182377657.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


At similar parameter count, the CNN wins because *local connectivity* +
*parameter sharing* match the structure of natural images. The MLP has no
inductive bias and treats the 784 pixels as 784 independent features.

## Self-test

<details>
<summary>Q1 — Input <code>(B, 3, 32, 32)</code>, <code>Conv2d(3, 64, kernel_size=3, padding=1)</code>. Output shape?</summary>

`(B, 64, 32, 32)`. Kernel 3 with padding 1 and stride 1 preserves the
spatial size.
</details>

<details>
<summary>Q2 — Why do CNNs have far fewer parameters than MLPs on images?</summary>

(1) Local connectivity — each output neuron looks at a small `k × k`
patch instead of the entire image. (2) Parameter sharing — the same
filter is applied at every spatial position. Both together cut parameters
by orders of magnitude.
</details>

<details>
<summary>Q3 — Why <code>AdaptiveAvgPool2d((1, 1))</code> before the final <code>Linear</code>?</summary>

It collapses any spatial size to `1×1`, so the classifier head accepts
any input resolution as long as the channel count matches.
</details>